## Import necessary python modules 

In [1]:
import cv2
import numpy as np
import time

from tpmodules.input_reader import VideoReader
from tpmodules.draw import draw_poses
from tpmodules.parse_poses import parse_poses
from argparse import ArgumentParser

#### Cannot load fast pose extraction, switched to legacy slow implementation. ####


## Cube definition

In [2]:
# Cube with roof (house)
cube_vertices = np.float32([[0, 0, 0], [0, 1, 0], [1, 1, 0], [1, 0, 0],
                       [0, 0, 1], [0, 1, 1], [1, 1, 1], [1, 0, 1],
                       [0, 0.5, 1.4], [1, 0.5, 1.4]])
cube_vertices -= 0.5 # Translate house to the center of the world
cube_side_length = 140
cube_vertices *= cube_side_length # Upscale house 
cube_edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7), (4, 8), (5, 8), (6, 9), (7, 9), (8, 9)]

## Parse input arguments for skeleton extraction

In [25]:
if __name__ == '__main__':
    parser = ArgumentParser(description='Mini project parser')
    parser.add_argument('-m', '--model', help='Required. Path to checkpoint with a trained model ' '(or an .xml file in case of OpenVINO inference).',
                        type=str, default='/opt/campux/lightweight-human-pose-estimation-3d-demo.pytorch/human-pose-estimation-3d.pth')
    parser.add_argument('--video', help='Optional. Path to camera id.', type=str, default='0')
    parser.add_argument('-d', '--device', help='Optional. Specify the target device to infer on: CPU or GPU. '
                             'The demo will look for a suitable plugin for device specified '
                             '(by default, it is GPU).',
                        type=str, default='GPU')
    parser.add_argument('--use-tensorrt', help='Optional. Run network with TensorRT as inference engine.',
                        action='store_true')
    parser.add_argument('--bg_image', help='Path to the background image.', type=str, default='data/office-background.png')
    parser.add_argument('--height-size', help='Network input layer height size (by default is set to 256).', type=int, default=256)
    parser.add_argument('--fx', type=np.float32, default=-1, help='Optional. Camera focal length.')
    args, unknown = parser.parse_known_args()

## Determining Input source 

In [26]:
frame_provider = VideoReader(args.video)
is_video = True
base_height = args.height_size
fx = args.fx

stride = 8
from modules.inference_engine_pytorch import InferenceEnginePyTorch
net = InferenceEnginePyTorch(args.model, args.device, use_tensorrt=args.use_tensorrt)

## Declaration of constants and keyboard mappings

In [43]:
constant_delay = 10 # In ms. Should be set empirically depending on computer frame rate
delay = constant_delay

# Letter-keyboard associations
esc_code = 27 # 'Esc' button
p_code = 112 # Letter 'p' button
draw_pose_code = 100 # Letter 'd' button
face_blur_code = 102 # Letter 'f' button
back_blur_code = 98 # Letter 'b' button
back_change_code = 115 # Letter 's' button
obj_hand_code = 111 # Letter 'o' button
space_code = 32

video_2d_window_name = 'Mini project with live input'
fps = 1.0/(constant_delay*0.001)

# Flags initialization
face_blur_flag = True  # Face blurring
back_blur_flag = False # Background blurring
back_change_flag = False# Background image change
obj_hand_flag = True # 3D object augmentation
draw_pose_flag = True # Draw extracted skeleton on the window

## Exercise area : Face blurring, background blurring, backround image change and 3D object addition

In [44]:
constant_delay = 1 # 

# For reference, skeleton joints are in the following order:
# ['centroid', neck', 'nose',  'l_sho', 'l_elb', 'l_wri', 'l_hip', 'l_knee', 'l_ank', 'r_sho', 'r_elb', 'r_wri', 'r_hip', 'r_knee', 'r_ank', 'r_eye', 'l_eye', 'r_ear', 'l_ear']

frame_count = 0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('data/live-output.mp4', fourcc, 10, (640, 360))

for frame in frame_provider:
    if frame is None: # If camera (or video) stop, exit the loop
        print('Frame is None because video/camera feed is empty or terminated')
        break
       
    frame_count += 1

    input_scale = base_height / frame.shape[0] # Scale factor used to resize images provided to the neural network.
    scaled_img = cv2.resize(frame, dsize=None, fx=input_scale, fy=input_scale)
    scaled_img = scaled_img[:, 0:scaled_img.shape[1] - (scaled_img.shape[1] % stride)]  # better to pad, but cut out for demo
    
    if fx < 0:  # Focal length is unknown
        fx = np.float32(0.8 * frame.shape[1])
    
    inference_result = net.infer(scaled_img) # Detect 3D skeleton using neural network
    
    # The following command retrieves the 3D poses of the skeleton joints and their 2D coordinates in the image plane
    poses_3d, poses_2d = parse_poses(inference_result, input_scale, stride, fx, is_video)
    
    final_frame = frame.copy()
    print(final_frame.shape)
    if poses_2d.size != 0 : # If skeleton/human exists in the current frame
        pose = np.array(poses_2d[0][0:-1]).reshape((-1, 3)).transpose()
        final_frame = frame.copy()
        mask =None
        #################
        # Face blurring # 
        #################
        
        if face_blur_flag == True:
            print('Face blurring')
            h, w = 100, 100
            x = int(pose[0,1])  
            y = int(pose[1,1])  
            x1 = max(0, x - w)
            x2 = min(frame.shape[1], x + w)
            y1 = max(0, y - h)
            y2 = min(frame.shape[0], y + h)
            
            mask = frame[y1:y2, x1:x2]

            if mask.size != 0:
                blurred_img = cv2.GaussianBlur(mask, (25,25), 10)
                final_frame[y1:y2,x1:x2 ] = blurred_img
            
                        

        #######################
        # Background blurring #
        #######################
        
        if back_blur_flag == True:
            print('Background blurring')
            ### ADD YOUR CODE ###

            mask=np.zeros(frame.shape[:2], dtype=np.uint8)
            draw_poses(mask,poses_2d)
            kernel=np.ones((100,100),np.uint8)
            mask=cv2.dilate(mask,kernel,iterations=1)
            blurred_img = cv2.GaussianBlur(frame, (25, 25), 10)
            final_frame = np.where(mask[:, :, np.newaxis] == 255, final_frame,blurred_img)
            
                
        ##########################
        # Background picture set #
        ##########################
        
        if back_change_flag == True:
            print('Background image change')
            ### ADD YOUR CODE ###
            mask=np.zeros(frame.shape[:2], dtype=np.uint8)
            draw_poses(mask,poses_2d)
            kernel=np.ones((125,125),np.uint8)
            mask=cv2.dilate(mask,kernel,iterations=1)
            bg_image=cv2.imread("data/office-background.png")
            bg_resized = cv2.resize(bg_image, (frame.shape[1], frame.shape[0]))
            final_frame = np.where(mask[:, :, np.newaxis] == (255,255,255), final_frame,bg_resized)
        ######################################    
        # Left-hand augmentation with object #
        ######################################
        
        if obj_hand_flag == True and pose[0, 5] != -1.0:
            print('3D object augmentation')

            x = int(pose[0, 5])
            y = int(pose[1, 5])
            fx = 593.47881447460395

            projected = []
            angle = np.radians(frame_count * 10)
            rvec = np.array([0, angle, 0], dtype=np.float32)
            R, _ = cv2.Rodrigues(rvec)

            for v in cube_vertices:
                vx = v[2]
                vy = v[1]
                vz = -v[0]
                vx2 = -vy
                vy2 = vx
                vz2 = vz
                v_rot = R @ np.array([vx2, vy2, vz2])
                scale = fx / (fx + v_rot[2])
                px = int(v_rot[0] * scale + x)
                py = int(-v_rot[1] * scale + y)
                projected.append((px, py))
            for (i, j) in cube_edges:
                cv2.line(final_frame, projected[i], projected[j], (255, 0, 0), 10)
            
        if(draw_pose_flag == True):
            draw_poses(final_frame, poses_2d) # Draw 2D skeleton on top of RGB video frames

    cv2.imshow(video_2d_window_name, final_frame)
    out.write(final_frame)
    
    key = cv2.waitKey(constant_delay) # Wait delay ms
    
    if key == esc_code: # Exit loop
        cv2.destroyAllWindows()
        break
        
    if key == face_blur_code: # face blurring
        face_blur_flag = not face_blur_flag
    if key == back_blur_code : # background blurring
        back_blur_flag = not back_blur_flag
    if key == back_change_code : # background photo change
        back_change_flag = not back_change_flag
    if key == obj_hand_code : # hand augmentation
        obj_hand_flag = not obj_hand_flag
    if key == draw_pose_code : # skeleton drawing
        draw_pose_flag = not draw_pose_flag
        
    if key == p_code: # Pause or unpause
        if delay == constant_delay:
            delay = 0
        else:
            delay = constant_delay
    if delay == 0 or not is_video:  # allow to rotate 3D canvas while on pause
        key = 0
        while (key != p_code and key != esc_code and key != space_code):
            key = cv2.waitKey(10)
        if key == esc_code:
            cv2.destroyAllWindows()
            break
        else:
            delay = constant_delay
frame_provider.cap.release()
out.release()
cv2.destroyAllWindows()
print("Exiting")

(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
3D object augmentation
(360, 640, 3)
Face blurring
(360, 640, 3)
Face blurring
3D object augmentation
(360, 640, 3)
Face blurring
3D object augmentation
(360, 640, 3)
Face blurring
3D object augmentation
(360, 640, 3